In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

from ADFWI.utils.assessment_metric import MAPE, MSE

base_path = "../Inductive_bias/imputation_bias/Marmousi2-nowater-Shots"

init_model = np.load(os.path.join(base_path,"data-shot=20/model/init_model.npz"))
true_model = np.load(os.path.join(base_path,"data-shot=20/model/true_model.npz"))
init_v   = init_model["vp"]
init_rho = init_model["rho"]
true_v   = true_model["vp"]
true_rho = true_model["rho"]

ox, oz  = 0, 0        
nz, nx  = 76, 200      
dx, dz  = 40, 40         
nt, dt  = 2500, 0.003     
nabc    = 30                 
x       = np.arange(nx)*dx/1000
z       = np.arange(nz)*dz/1000
x_mesh,z_mesh = np.meshgrid(x,z)

def get_observed_sys(shot_num,rcv_num):
    # Source
    if shot_num == 40:
        # 40 sources
        src_z = np.array([1 for i in range(2, nx-1, 5)])*dz/1000  # Z-coordinates for sources
        src_x = np.array([i for i in range(2, nx-1, 5)])*dz/1000  # X-coordinates for sources
    elif shot_num == 20:
        # 20 sources
        src_z = np.array([1 for i in range(5, nx-1, 10)])*dz/1000  # Z-coordinates for sources
        src_x = np.array([i for i in range(5, nx-1, 10)])*dz/1000  # X-coordinates for sources
    elif shot_num == 10:
        # 10 sources
        src_z = np.array([1 for i in range(10, nx-1, 20)])*dz/1000  # Z-coordinates for sources
        src_x = np.array([i for i in range(10, nx-1, 20)])*dz/1000  # X-coordinates for sources
    elif shot_num == 5:
        # 5 sources
        src_z = np.array([1 for i in range(20, nx-1, 40)])*dz/1000  # Z-coordinates for sources
        src_x = np.array([i for i in range(20, nx-1, 40)])*dz/1000  # X-coordinates for sources
    elif shot_num == 2:
        # 2 sources
        src_z = np.array([1 for i in range(60, nx-1, 80)])*dz/1000  # Z-coordinates for sources
        src_x = np.array([i for i in range(60, nx-1, 80)])*dz/1000  # X-coordinates for sources
        
    # receiver
    if rcv_num == 200:
    # src = 200
        rcv_z = np.array([1 for i in range(0, nx, 1)])*dz/1000  # Z-coordinates for receivers
        rcv_x = np.array([j for j in range(0, nx, 1)])*dz/1000  # X-coordinates for receivers
    elif rcv_num == 50:
    # src = 50
        rcv_z = np.array([1 for i in range(0, nx, 4)])*dz/1000  # Z-coordinates for receivers
        rcv_x = np.array([j for j in range(0, nx, 4)])*dz/1000  # X-coordinates for receivers
    elif rcv_num == 20:
    # src = 20
        rcv_z = np.array([1 for i in range(5, nx, 10)])*dz/1000  # Z-coordinates for receivers
        rcv_x = np.array([j for j in range(5, nx, 10)])*dz/1000  # X-coordinates for receivers
    return src_x,src_z,rcv_x,rcv_z

shot = 2
src_x,src_z,rcv_x,rcv_z = get_observed_sys(shot_num=2,rcv_num=200)

vmin = true_v.min();vmax = true_v.max()  

MAX_ITER = 300

In [ ]:
import matplotlib.transforms as mtransforms
from scipy.interpolate import griddata

def plot_vel_single_for_all(fig,ax,v,title="",MSE="",vmin=None,vmax=None,cmap = "rainbow"):
    plt.rc('font',family='Times New Roman')
    # plm = ax.pcolormesh(x_mesh, z_mesh, v,cmap=cmap,vmin=vmin,vmax=vmax)
    x = np.arange(nx*3)*dx/3/1000
    z = np.arange(nz*3)*dz/3/1000
    x_mesh_new,z_mesh_new = np.meshgrid(x,z)

    v_new = griddata((x_mesh.flatten(), z_mesh.flatten()), v.flatten(), (x_mesh_new, z_mesh_new), method='cubic')
    
    plm = ax.pcolormesh(x_mesh_new, z_mesh_new, v_new,cmap=cmap,vmin=vmin,vmax=vmax,shading="nearest")
    ax.invert_yaxis()
    ax.tick_params(labelsize = 14)
    ax.set_title(title,fontsize=14)
    ax.text(0.2,0.4,MSE,fontsize=14,c="w")
    return plm

def add_right_cax(ax, pad, width):
    axpos = ax.get_position()
    caxpos = mtransforms.Bbox.from_extents(
        axpos.x1 + pad,
        axpos.y0,
        axpos.x1 + pad + width,
        axpos.y1
    )
    cax = ax.figure.add_axes(caxpos)

    return cax

def add_bottom_cax(ax, pad, height):
    axpos = ax.get_position()
    caxpos = mtransforms.Bbox.from_extents(
        axpos.x0,
        axpos.y0 - pad - height,
        axpos.x1,
        axpos.y0 - pad
    )
    cax = ax.figure.add_axes(caxpos)
    
    return cax

def plot_vel_singleline_for_all(ax,v_true,v_init,v_inv,x_distance,title,show_xlabel=True,show_ylabel=False,show_legend=False):
    ax.plot(v_true[:,int(x_distance//dx)]/1000,  z, c='k',   linewidth=2, linestyle="-" ,label="True")
    ax.plot(v_init[:,int(x_distance//dx)]/1000,  z, c='gray',linewidth=2, linestyle="-" ,label="Init")
    ax.plot(v_inv [:,int(x_distance//dx)]/1000,  z, c='r',   linewidth=2, linestyle="--",label="Inverted")
    ax.tick_params(labelsize = 12)
    if not show_xlabel:
        ax.set_xticks([])

    if not show_ylabel:
        ax.set_yticks([])
    else:
        ax.tick_params(labelsize = 10)
    ax.invert_yaxis()
    
    if show_legend:
        ax.legend(fontsize = 12)
    ax.set_title(title,fontsize=12)

## Varing Shots

In [ ]:
itervp_baseline0    = np.load(os.path.join(base_path,"data-shot=20/inversion-vp-baseline/iter_vp.npz"))["data"][:MAX_ITER]
itervp_baseline1    = np.load(os.path.join(base_path,"data-shot=10/inversion-vp-baseline/iter_vp.npz"))["data"][:MAX_ITER]
itervp_baseline2    = np.load(os.path.join(base_path,"data-shot=5/inversion-vp-baseline/iter_vp.npz"))["data"][:MAX_ITER]
itervp_baseline3    = np.load(os.path.join(base_path,"data-shot=2/inversion-vp-baseline/iter_vp.npz"))["data"][:MAX_ITER]

itervp_CNN_1x64_0   = np.load(os.path.join(base_path,"data-shot=20/inversion-vp-CNN-1x64/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_1x64_1   = np.load(os.path.join(base_path,"data-shot=10/inversion-vp-CNN-1x64/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_1x64_2   = np.load(os.path.join(base_path,"data-shot=5/inversion-vp-CNN-1x64/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_1x64_3   = np.load(os.path.join(base_path,"data-shot=2/inversion-vp-CNN-1x64/iter_vp.npz"))["data"][:MAX_ITER]

itervp_CNN_1x128_0   = np.load(os.path.join(base_path,"data-shot=20/inversion-vp-CNN-1x128/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_1x128_1   = np.load(os.path.join(base_path,"data-shot=10/inversion-vp-CNN-1x128/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_1x128_2   = np.load(os.path.join(base_path,"data-shot=5/inversion-vp-CNN-1x128/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_1x128_3   = np.load(os.path.join(base_path,"data-shot=2/inversion-vp-CNN-1x128/iter_vp.npz"))["data"][:MAX_ITER]

itervp_CNN_1x256_0   = np.load(os.path.join(base_path,"data-shot=20/inversion-vp-CNN-1x256/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_1x256_1   = np.load(os.path.join(base_path,"data-shot=10/inversion-vp-CNN-1x256/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_1x256_2   = np.load(os.path.join(base_path,"data-shot=5/inversion-vp-CNN-1x256/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_1x256_3   = np.load(os.path.join(base_path,"data-shot=2/inversion-vp-CNN-1x256/iter_vp.npz"))["data"][:MAX_ITER]

itervp_CNN_1x512_0   = np.load(os.path.join(base_path,"data-shot=20/inversion-vp-CNN-1x512/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_1x512_1   = np.load(os.path.join(base_path,"data-shot=10/inversion-vp-CNN-1x512/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_1x512_2   = np.load(os.path.join(base_path,"data-shot=5/inversion-vp-CNN-1x512/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_1x512_3   = np.load(os.path.join(base_path,"data-shot=2/inversion-vp-CNN-1x512/iter_vp.npz"))["data"][:MAX_ITER]


itervp_CNN_2x64_0  = np.load(os.path.join(base_path,"data-shot=20/inversion-vp-CNN-2x64/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_2x64_1  = np.load(os.path.join(base_path,"data-shot=10/inversion-vp-CNN-2x64/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_2x64_2  = np.load(os.path.join(base_path,"data-shot=5/inversion-vp-CNN-2x64/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_2x64_3  = np.load(os.path.join(base_path,"data-shot=2/inversion-vp-CNN-2x64/iter_vp.npz"))["data"][:MAX_ITER]

itervp_CNN_2x128_0  = np.load(os.path.join(base_path,"data-shot=20/inversion-vp-CNN-2x128/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_2x128_1  = np.load(os.path.join(base_path,"data-shot=10/inversion-vp-CNN-2x128/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_2x128_2  = np.load(os.path.join(base_path,"data-shot=5/inversion-vp-CNN-2x128/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_2x128_3  = np.load(os.path.join(base_path,"data-shot=2/inversion-vp-CNN-2x128/iter_vp.npz"))["data"][:MAX_ITER]

itervp_CNN_2x256_0  = np.load(os.path.join(base_path,"data-shot=20/inversion-vp-CNN-2x256/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_2x256_1  = np.load(os.path.join(base_path,"data-shot=10/inversion-vp-CNN-2x256/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_2x256_2  = np.load(os.path.join(base_path,"data-shot=5/inversion-vp-CNN-2x256/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_2x256_3  = np.load(os.path.join(base_path,"data-shot=2/inversion-vp-CNN-2x256/iter_vp.npz"))["data"][:MAX_ITER]

itervp_CNN_2x512_0  = np.load(os.path.join(base_path,"data-shot=20/inversion-vp-CNN-2x512/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_2x512_1  = np.load(os.path.join(base_path,"data-shot=10/inversion-vp-CNN-2x512/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_2x512_2  = np.load(os.path.join(base_path,"data-shot=5/inversion-vp-CNN-2x512/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_2x512_3  = np.load(os.path.join(base_path,"data-shot=2/inversion-vp-CNN-2x512/iter_vp.npz"))["data"][:MAX_ITER]

itervp_CNN_3x64_0  = np.load(os.path.join(base_path,"data-shot=20/inversion-vp-CNN-3x64/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_3x64_1  = np.load(os.path.join(base_path,"data-shot=10/inversion-vp-CNN-3x64/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_3x64_2  = np.load(os.path.join(base_path,"data-shot=5/inversion-vp-CNN-3x64/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_3x64_3  = np.load(os.path.join(base_path,"data-shot=2/inversion-vp-CNN-3x64/iter_vp.npz"))["data"][:MAX_ITER]

itervp_CNN_3x128_0  = np.load(os.path.join(base_path,"data-shot=20/inversion-vp-CNN-3x128/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_3x128_1  = np.load(os.path.join(base_path,"data-shot=10/inversion-vp-CNN-3x128/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_3x128_2  = np.load(os.path.join(base_path,"data-shot=5/inversion-vp-CNN-3x128/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_3x128_3  = np.load(os.path.join(base_path,"data-shot=2/inversion-vp-CNN-3x128/iter_vp.npz"))["data"][:MAX_ITER]

itervp_CNN_3x256_0  = np.load(os.path.join(base_path,"data-shot=20/inversion-vp-CNN-3x256/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_3x256_1  = np.load(os.path.join(base_path,"data-shot=10/inversion-vp-CNN-3x256/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_3x256_2  = np.load(os.path.join(base_path,"data-shot=5/inversion-vp-CNN-3x256/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_3x256_3  = np.load(os.path.join(base_path,"data-shot=2/inversion-vp-CNN-3x256/iter_vp.npz"))["data"][:MAX_ITER]

itervp_CNN_3x512_0  = np.load(os.path.join(base_path,"data-shot=20/inversion-vp-CNN-3x512/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_3x512_1  = np.load(os.path.join(base_path,"data-shot=10/inversion-vp-CNN-3x512/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_3x512_2  = np.load(os.path.join(base_path,"data-shot=5/inversion-vp-CNN-3x512/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_3x512_3  = np.load(os.path.join(base_path,"data-shot=2/inversion-vp-CNN-3x512/iter_vp.npz"))["data"][:MAX_ITER]

In [ ]:
iterloss_baseline0  = np.load(os.path.join(base_path,"data-shot=20/inversion-vp-baseline/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_baseline1  = np.load(os.path.join(base_path,"data-shot=10/inversion-vp-baseline/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_baseline2  = np.load(os.path.join(base_path,"data-shot=5/inversion-vp-baseline/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_baseline3  = np.load(os.path.join(base_path,"data-shot=2/inversion-vp-baseline/iter_loss.npz"))["data"][:MAX_ITER]

iterloss_1x64_CNN0 = np.load(os.path.join(base_path,"data-shot=20/inversion-vp-CNN-1x64/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_1x64_CNN1 = np.load(os.path.join(base_path,"data-shot=10/inversion-vp-CNN-1x64/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_1x64_CNN2 = np.load(os.path.join(base_path,"data-shot=5/inversion-vp-CNN-1x64/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_1x64_CNN3 = np.load(os.path.join(base_path,"data-shot=2/inversion-vp-CNN-1x64/iter_loss.npz"))["data"][:MAX_ITER]

iterloss_1x128_CNN0 = np.load(os.path.join(base_path,"data-shot=20/inversion-vp-CNN-1x128/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_1x128_CNN1 = np.load(os.path.join(base_path,"data-shot=10/inversion-vp-CNN-1x128/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_1x128_CNN2 = np.load(os.path.join(base_path,"data-shot=5/inversion-vp-CNN-1x128/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_1x128_CNN3 = np.load(os.path.join(base_path,"data-shot=2/inversion-vp-CNN-1x128/iter_loss.npz"))["data"][:MAX_ITER]

iterloss_1x256_CNN0 = np.load(os.path.join(base_path,"data-shot=20/inversion-vp-CNN-1x256/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_1x256_CNN1 = np.load(os.path.join(base_path,"data-shot=10/inversion-vp-CNN-1x256/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_1x256_CNN2 = np.load(os.path.join(base_path,"data-shot=5/inversion-vp-CNN-1x256/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_1x256_CNN3 = np.load(os.path.join(base_path,"data-shot=2/inversion-vp-CNN-1x256/iter_loss.npz"))["data"][:MAX_ITER]

iterloss_1x512_CNN0 = np.load(os.path.join(base_path,"data-shot=20/inversion-vp-CNN-1x512/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_1x512_CNN1 = np.load(os.path.join(base_path,"data-shot=10/inversion-vp-CNN-1x512/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_1x512_CNN2 = np.load(os.path.join(base_path,"data-shot=5/inversion-vp-CNN-1x512/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_1x512_CNN3 = np.load(os.path.join(base_path,"data-shot=2/inversion-vp-CNN-1x512/iter_loss.npz"))["data"][:MAX_ITER]


iterloss_2x64_CNN0 = np.load(os.path.join(base_path,"data-shot=20/inversion-vp-CNN-2x64/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_2x64_CNN1 = np.load(os.path.join(base_path,"data-shot=10/inversion-vp-CNN-2x64/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_2x64_CNN2 = np.load(os.path.join(base_path,"data-shot=5/inversion-vp-CNN-2x64/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_2x64_CNN3 = np.load(os.path.join(base_path,"data-shot=2/inversion-vp-CNN-2x64/iter_loss.npz"))["data"][:MAX_ITER]

iterloss_2x128_CNN0 = np.load(os.path.join(base_path,"data-shot=20/inversion-vp-CNN-2x128/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_2x128_CNN1 = np.load(os.path.join(base_path,"data-shot=10/inversion-vp-CNN-2x128/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_2x128_CNN2 = np.load(os.path.join(base_path,"data-shot=5/inversion-vp-CNN-2x128/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_2x128_CNN3 = np.load(os.path.join(base_path,"data-shot=2/inversion-vp-CNN-2x128/iter_loss.npz"))["data"][:MAX_ITER]

iterloss_2x256_CNN0 = np.load(os.path.join(base_path,"data-shot=20/inversion-vp-CNN-2x256/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_2x256_CNN1 = np.load(os.path.join(base_path,"data-shot=10/inversion-vp-CNN-2x256/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_2x256_CNN2 = np.load(os.path.join(base_path,"data-shot=5/inversion-vp-CNN-2x256/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_2x256_CNN3 = np.load(os.path.join(base_path,"data-shot=2/inversion-vp-CNN-2x256/iter_loss.npz"))["data"][:MAX_ITER]

iterloss_2x512_CNN0 = np.load(os.path.join(base_path,"data-shot=20/inversion-vp-CNN-2x512/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_2x512_CNN1 = np.load(os.path.join(base_path,"data-shot=10/inversion-vp-CNN-2x512/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_2x512_CNN2 = np.load(os.path.join(base_path,"data-shot=5/inversion-vp-CNN-2x512/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_2x512_CNN3 = np.load(os.path.join(base_path,"data-shot=2/inversion-vp-CNN-2x512/iter_loss.npz"))["data"][:MAX_ITER]


iterloss_3x64_CNN0 = np.load(os.path.join(base_path,"data-shot=20/inversion-vp-CNN-3x64/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_3x64_CNN1 = np.load(os.path.join(base_path,"data-shot=10/inversion-vp-CNN-3x64/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_3x64_CNN2 = np.load(os.path.join(base_path,"data-shot=5/inversion-vp-CNN-3x64/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_3x64_CNN3 = np.load(os.path.join(base_path,"data-shot=2/inversion-vp-CNN-3x64/iter_loss.npz"))["data"][:MAX_ITER]

iterloss_3x128_CNN0 = np.load(os.path.join(base_path,"data-shot=20/inversion-vp-CNN-3x128/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_3x128_CNN1 = np.load(os.path.join(base_path,"data-shot=10/inversion-vp-CNN-3x128/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_3x128_CNN2 = np.load(os.path.join(base_path,"data-shot=5/inversion-vp-CNN-3x128/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_3x128_CNN3 = np.load(os.path.join(base_path,"data-shot=2/inversion-vp-CNN-3x128/iter_loss.npz"))["data"][:MAX_ITER]

iterloss_3x256_CNN0 = np.load(os.path.join(base_path,"data-shot=20/inversion-vp-CNN-3x256/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_3x256_CNN1 = np.load(os.path.join(base_path,"data-shot=10/inversion-vp-CNN-3x256/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_3x256_CNN2 = np.load(os.path.join(base_path,"data-shot=5/inversion-vp-CNN-3x256/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_3x256_CNN3 = np.load(os.path.join(base_path,"data-shot=2/inversion-vp-CNN-3x256/iter_loss.npz"))["data"][:MAX_ITER]

iterloss_3x512_CNN0 = np.load(os.path.join(base_path,"data-shot=20/inversion-vp-CNN-3x512/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_3x512_CNN1 = np.load(os.path.join(base_path,"data-shot=10/inversion-vp-CNN-3x512/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_3x512_CNN2 = np.load(os.path.join(base_path,"data-shot=5/inversion-vp-CNN-3x512/iter_loss.npz"))["data"][:MAX_ITER]
iterloss_3x512_CNN3 = np.load(os.path.join(base_path,"data-shot=2/inversion-vp-CNN-3x512/iter_loss.npz"))["data"][:MAX_ITER]

In [ ]:
# 计算 MAPE 误差
WIN_SIZE = 3

baseline_losses_shots = [
    np.min(MAPE(true_v=true_v, inv_v=itervp_baseline0)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_baseline1)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_baseline2)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_baseline3)),
]

cnn_losses1x64 = [
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x64_0)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x64_1)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x64_2)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x64_3)),
]

cnn_losses1x128 = [
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x128_0)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x128_1)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x128_2)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x128_3)),
]

cnn_losses1x256 = [
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x256_0)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x256_1)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x256_2)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x256_3)),
]

cnn_losses1x512 = [
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x512_0)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x512_1)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x512_2)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x512_3)),
]

cnn_losses2x64 = [
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x64_0)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x64_1)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x64_2)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x64_3)),
]

cnn_losses2x128 = [
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x128_0)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x128_1)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x128_2)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x128_3)),
]

cnn_losses2x256 = [
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x256_0)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x256_1)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x256_2)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x256_3)),
]

cnn_losses2x512 = [
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x512_0)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x512_1)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x512_2)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x512_3)),
]

cnn_losses3x64 = [
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x64_0)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x64_1)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x64_2)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x64_3)),
]

cnn_losses3x128 = [
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x128_0)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x128_1)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x128_2)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x128_3)),
]

cnn_losses3x256 = [
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x256_0)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x256_1)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x256_2)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x256_3)),
]

cnn_losses3x512 = [
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x512_0)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x512_1)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x512_2)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x512_3)),
]

cnn_losses_shots = np.array([
    cnn_losses1x64, cnn_losses1x128, cnn_losses1x256,
    cnn_losses2x64, cnn_losses2x128, cnn_losses2x256,
    cnn_losses3x64, cnn_losses3x128, cnn_losses3x256
])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
colors = sns.color_palette("muted")

xlist = np.array([2, 5, 10, 20])
xtick_labels = [2, 5, 10, 20]

cnn_losses = np.array([
    cnn_losses1x64, cnn_losses1x128, cnn_losses1x256,
    cnn_losses2x64, cnn_losses2x128, cnn_losses2x256,
    cnn_losses3x64, cnn_losses3x128, cnn_losses3x256
])

cnn_losses = np.sort(cnn_losses,axis=0)
cnn_losses = cnn_losses[:8,:]

cnn_mean = np.mean(cnn_losses, axis=0)
cnn_std = np.std(cnn_losses, axis=0)

# construct the plot
plt.figure(figsize=(8, 4))

plt.plot(xlist, baseline_losses_shots[::-1], marker='o', linestyle='-', color=colors[0], 
         label="FWI (w/o Reparameterization)", linewidth=2.5, markersize=7)
plt.fill_between(xlist, (cnn_mean - cnn_std)[::-1], (cnn_mean + cnn_std)[::-1], color=colors[1], alpha=0.3)
plt.plot(xlist, cnn_mean[::-1], marker='s', linestyle='-', color="crimson" , label="FWI (w/ CNN Reparameterization)", linewidth=2.5, markersize=7)
plt.xlabel("Shots Number", fontsize=12)
plt.ylabel("MAPE", fontsize=12)
plt.xticks(xlist, xtick_labels, fontsize=12)
plt.yticks(fontsize=11)
plt.legend(frameon=False, fontsize=12)
plt.grid(True, linestyle="--", alpha=0.6)
# sns.despine(left=False, bottom=False)
plt.show()


## Varing Receivers

In [ ]:
base_path = "../Inductive_bias/imputation_bias/Marmousi2-nowater-Receiver"

itervp_baseline0    = np.load(os.path.join(base_path,"data-shot=10_rcv=20/inversion-vp-baseline/iter_vp.npz"))["data"][:MAX_ITER]
itervp_baseline1    = np.load(os.path.join(base_path,"data-shot=10_rcv=50/inversion-vp-baseline/iter_vp.npz"))["data"][:MAX_ITER]
itervp_baseline2    = np.load(os.path.join(base_path,"data-shot=10_rcv=200/inversion-vp-baseline/iter_vp.npz"))["data"][:MAX_ITER]

itervp_CNN_1x64_0   = np.load(os.path.join(base_path,"data-shot=10_rcv=20/inversion-vp-CNN-1x64/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_1x64_1   = np.load(os.path.join(base_path,"data-shot=10_rcv=50/inversion-vp-CNN-1x64/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_1x64_2   = np.load(os.path.join(base_path,"data-shot=10_rcv=200/inversion-vp-CNN-1x64/iter_vp.npz"))["data"][:MAX_ITER]

itervp_CNN_1x128_0   = np.load(os.path.join(base_path,"data-shot=10_rcv=20/inversion-vp-CNN-1x128/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_1x128_1   = np.load(os.path.join(base_path,"data-shot=10_rcv=50/inversion-vp-CNN-1x128/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_1x128_2   = np.load(os.path.join(base_path,"data-shot=10_rcv=200/inversion-vp-CNN-1x128/iter_vp.npz"))["data"][:MAX_ITER]

itervp_CNN_1x256_0   = np.load(os.path.join(base_path,"data-shot=10_rcv=20/inversion-vp-CNN-1x256/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_1x256_1   = np.load(os.path.join(base_path,"data-shot=10_rcv=50/inversion-vp-CNN-1x256/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_1x256_2   = np.load(os.path.join(base_path,"data-shot=10_rcv=200/inversion-vp-CNN-1x256/iter_vp.npz"))["data"][:MAX_ITER]

itervp_CNN_1x512_0   = np.load(os.path.join(base_path,"data-shot=10_rcv=20/inversion-vp-CNN-1x512/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_1x512_1   = np.load(os.path.join(base_path,"data-shot=10_rcv=50/inversion-vp-CNN-1x512/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_1x512_2   = np.load(os.path.join(base_path,"data-shot=10_rcv=200/inversion-vp-CNN-1x512/iter_vp.npz"))["data"][:MAX_ITER]


itervp_CNN_2x64_0  = np.load(os.path.join(base_path,"data-shot=10_rcv=20/inversion-vp-CNN-2x64/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_2x64_1  = np.load(os.path.join(base_path,"data-shot=10_rcv=50/inversion-vp-CNN-2x64/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_2x64_2  = np.load(os.path.join(base_path,"data-shot=10_rcv=200/inversion-vp-CNN-2x64/iter_vp.npz"))["data"][:MAX_ITER]

itervp_CNN_2x128_0  = np.load(os.path.join(base_path,"data-shot=10_rcv=20/inversion-vp-CNN-2x128/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_2x128_1  = np.load(os.path.join(base_path,"data-shot=10_rcv=50/inversion-vp-CNN-2x128/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_2x128_2  = np.load(os.path.join(base_path,"data-shot=10_rcv=200/inversion-vp-CNN-2x128/iter_vp.npz"))["data"][:MAX_ITER]

itervp_CNN_2x256_0  = np.load(os.path.join(base_path,"data-shot=10_rcv=20/inversion-vp-CNN-2x256/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_2x256_1  = np.load(os.path.join(base_path,"data-shot=10_rcv=50/inversion-vp-CNN-2x256/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_2x256_2  = np.load(os.path.join(base_path,"data-shot=10_rcv=200/inversion-vp-CNN-2x256/iter_vp.npz"))["data"][:MAX_ITER]

itervp_CNN_2x512_0  = np.load(os.path.join(base_path,"data-shot=10_rcv=20/inversion-vp-CNN-2x512/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_2x512_1  = np.load(os.path.join(base_path,"data-shot=10_rcv=50/inversion-vp-CNN-2x512/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_2x512_2  = np.load(os.path.join(base_path,"data-shot=10_rcv=200/inversion-vp-CNN-2x512/iter_vp.npz"))["data"][:MAX_ITER]

itervp_CNN_3x64_0  = np.load(os.path.join(base_path,"data-shot=10_rcv=20/inversion-vp-CNN-3x64/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_3x64_1  = np.load(os.path.join(base_path,"data-shot=10_rcv=50/inversion-vp-CNN-3x64/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_3x64_2  = np.load(os.path.join(base_path,"data-shot=10_rcv=200/inversion-vp-CNN-3x64/iter_vp.npz"))["data"][:MAX_ITER]

itervp_CNN_3x128_0  = np.load(os.path.join(base_path,"data-shot=10_rcv=20/inversion-vp-CNN-3x128/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_3x128_1  = np.load(os.path.join(base_path,"data-shot=10_rcv=50/inversion-vp-CNN-3x128/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_3x128_2  = np.load(os.path.join(base_path,"data-shot=10_rcv=200/inversion-vp-CNN-3x128/iter_vp.npz"))["data"][:MAX_ITER]

itervp_CNN_3x256_0  = np.load(os.path.join(base_path,"data-shot=10_rcv=20/inversion-vp-CNN-3x256/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_3x256_1  = np.load(os.path.join(base_path,"data-shot=10_rcv=50/inversion-vp-CNN-3x256/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_3x256_2  = np.load(os.path.join(base_path,"data-shot=10_rcv=200/inversion-vp-CNN-3x256/iter_vp.npz"))["data"][:MAX_ITER]

itervp_CNN_3x512_0  = np.load(os.path.join(base_path,"data-shot=10_rcv=20/inversion-vp-CNN-3x512/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_3x512_1  = np.load(os.path.join(base_path,"data-shot=10_rcv=50/inversion-vp-CNN-3x512/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_3x512_2  = np.load(os.path.join(base_path,"data-shot=10_rcv=200/inversion-vp-CNN-3x512/iter_vp.npz"))["data"][:MAX_ITER]

In [ ]:
# calculate the MAPE error
WIN_SIZE = 3

baseline_losses_receiver = [
    np.min(MAPE(true_v=true_v, inv_v=itervp_baseline0)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_baseline1)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_baseline2)),
]

cnn_losses1x64 = [
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x64_0)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x64_1)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x64_2)),
]

cnn_losses1x128 = [
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x128_0)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x128_1)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x128_2)),
]

cnn_losses1x256 = [
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x256_0)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x256_1)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x256_2)),
]

cnn_losses1x512 = [
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x512_0)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x512_1)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_1x512_2)),
]

cnn_losses2x64 = [
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x64_0)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x64_1)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x64_2)),
]

cnn_losses2x128 = [
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x128_0)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x128_1)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x128_2)),
]

cnn_losses2x256 = [
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x256_0)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x256_1)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x256_2)),
]

cnn_losses2x512 = [
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x512_0)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x512_1)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_2x512_2)),
]

cnn_losses3x64 = [
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x64_0)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x64_1)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x64_2)),
]

cnn_losses3x128 = [
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x128_0)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x128_1)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x128_2)),
]

cnn_losses3x256 = [
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x256_0)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x256_1)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x256_2)),
]

cnn_losses3x512 = [
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x512_0)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x512_1)),
    np.min(MAPE(true_v=true_v, inv_v=itervp_CNN_3x512_2)),
]

cnn_losses_receivers = np.array([
    cnn_losses1x64, cnn_losses1x128, cnn_losses1x256,
    cnn_losses2x64, cnn_losses2x128, cnn_losses2x256,
    cnn_losses3x64, cnn_losses3x128, cnn_losses3x256
])


In [ ]:
np.argmin(cnn_losses_shots,axis=0)

In [ ]:
np.argmin(cnn_losses_receivers,axis=0)

## Article Figure (MAPE)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec
import seaborn as sns
from matplotlib.ticker import MaxNLocator
plt.rcParams['svg.fonttype'] = 'none'
sns.set_style("ticks")  # 或者使用 "darkgrid", "ticks", "white", "dark"

palette = sns.color_palette("deep")
colors = {
    "baseline": "#4D4D4D",  # Dark gray
    "cnn_dv": palette[0],    # Blue for CNN-v
    "cnn_v": sns.dark_palette(palette[0], 2)[1],  # Darker blue for CNN-dv
    "mlp_dv": palette[1],    # Green for MLP-v
    "mlp_v": sns.dark_palette(palette[1], 2)[1],  # Darker green for MLP-dv
    "unet_dv": palette[2],   # Purple for Unet-v
    "unet_v": sns.dark_palette(palette[2], 2)[1]  # Darker purple for Unet-dv
}

def plot_observed_sys(axs, shot_num=2, rcv_num=200, label=False):
    """ Plot observed system with sources and receivers """
    src_x, src_z, rcv_x, rcv_z = get_observed_sys(shot_num, rcv_num=rcv_num)
    scatters = []
    labels = []
    
    for i in range(shot_num):
        scatter_rcv = axs.scatter(rcv_x, rcv_z + i, color='k', marker='v', s=1, label='Receivers' if label and i == 0 else None)  # Receivers
        scatter_src = axs.scatter(src_x[i], src_z[i] + i, color='red', marker='*', s=100, label='Sources' if label and i == 0 else None)  # Sources
        
        if label and i == 0:
            scatters.extend([scatter_src, scatter_rcv])
            labels.extend(['Sources','Receivers'])

    # axs.set_xticks([])
    axs.xaxis.set_major_locator(MaxNLocator(3))
    axs.set_yticks([])
    axs.tick_params(labelsize=12)
    if shot_num == 2:
        axs.set_ylim(-2, 3)

    return scatters, labels

# Create the figure and define grid layout
fig = plt.figure(figsize=(15, 10))
gs = gridspec.GridSpec(3, 28, height_ratios=[1, 1.2, 1])

# Generate subplots
axes = [fig.add_subplot(gs[0, i * 4:(i + 1) * 4]) for i in range(7)]
shot_nums = [2, 5, 10, 20, 10, 10, 10]
rcv_nums = [200, 200, 200, 200, 20, 50, 200]

# Plot observed system and collect legend handles
legend_handles, legend_labels = None, None

for i, ax in enumerate(axes):
    handles, labels = plot_observed_sys(ax, shot_num=shot_nums[i], rcv_num=rcv_nums[i], label=(i == 0))
    ax.set_title(r"$N_s$={},$N_r$={}".format(shot_nums[i], rcv_nums[i]), fontsize=12)
    
    if i == 0:  # Save legend handles from the first subplot
        legend_handles, legend_labels = handles, labels
        ax.set_ylabel("Shots Index", fontsize=14)
# Add a shared legend at the bottom of the first row
fig.legend(legend_handles, legend_labels, loc='lower center', bbox_to_anchor=(0.5, 0.61), ncol=2, fontsize=12, frameon=False)

################################################################
axes1 = fig.add_subplot(gs[1, :16])
cnn_losses = np.sort(cnn_losses_shots,axis=0)
cnn_losses = cnn_losses[:6,:]
cnn_mean   = np.mean(cnn_losses, axis=0)
cnn_std    = np.std(cnn_losses, axis=0)
xlist = np.array([2, 5, 10, 20])
xtick_labels = [2, 5, 10, 20]
axes1.plot(xlist, baseline_losses_shots[::-1], marker='o', linestyle='-', color=colors['baseline'], label="Traditional FWI", linewidth=2.5, markersize=7)
axes1.fill_between(xlist, (cnn_mean - cnn_std)[::-1], (cnn_mean + cnn_std)[::-1], color=colors['cnn_v'], alpha=0.3)
axes1.plot(xlist, cnn_mean[::-1], marker='s', linestyle='-', color=colors['cnn_v'] , label=r"CNN-$v_s$", linewidth=2.5, markersize=7)
axes1.set_xticks(xlist, xtick_labels, fontsize=14)
axes1.tick_params(labelsize=14)
axes1.set_xlabel("Shots Number", fontsize=14)
axes1.set_ylabel("MAPE", fontsize=14)
axes1.set_ylim(4,9.5)
axes1.legend(frameon=False, fontsize=14,loc='upper right')
axes1.grid(True, linestyle="--", alpha=0.6)

axes2 = fig.add_subplot(gs[1, 16:])
cnn_losses = np.sort(cnn_losses_receivers,axis=0)
cnn_losses = cnn_losses[:8,:]
cnn_mean   = np.mean(cnn_losses, axis=0)
cnn_std    = np.std(cnn_losses, axis=0)
xlist      = np.array([0, 1, 2])
xtick_labels = [20, 50, 200]
axes2.plot(xlist, baseline_losses_receiver, marker='o', linestyle='-', color=colors['baseline'], label="Traditional FWI", linewidth=2.5, markersize=7)
axes2.fill_between(xlist, (cnn_mean - cnn_std), (cnn_mean + cnn_std), color=colors['cnn_v'], alpha=0.3)
axes2.plot(xlist, cnn_mean, marker='s', linestyle='-', color=colors['cnn_v'] , label=r"CNN-$v_s$", linewidth=2.5, markersize=7)
axes2.set_xticks(xlist, xtick_labels, fontsize=14)
axes2.set_xlabel("Receiver Number", fontsize=14)
axes2.tick_params(axis='y', which='both', left=False, right=False)
axes2.set_yticklabels([])
axes2.set_ylim(4, 9.5)
axes2.legend(frameon=False, fontsize=14,loc='upper right')
axes2.grid(True, linestyle="--", alpha=0.6)

##################################################################
axes21 = fig.add_subplot(gs[2, 1:13])
axes22 = fig.add_subplot(gs[2, 15:-1])

base_path = "../Inductive_bias/imputation_bias/Marmousi2-nowater-Receiver"
itervp_baseline2    = np.load(os.path.join(base_path,"data-shot=10_rcv=20/inversion-vp-baseline/iter_vp.npz"))["data"][:MAX_ITER]
itervp_CNN_2x256_2  = np.load(os.path.join(base_path,"data-shot=10_rcv=20/inversion-vp-CNN-2x256/iter_vp.npz"))["data"][:MAX_ITER]

im1 = plot_vel_single_for_all(fig,axes21,itervp_baseline2[-1]      ,title=None   ,MSE=r"Traditional FWI  MAPE:" + " " + r"{:.2f}".format(MAPE(true_v,itervp_baseline2[-1])))
im2 = plot_vel_single_for_all(fig,axes22,itervp_CNN_2x256_2[-1]    ,title=None   ,MSE=r"CNN-$v_p$ MAPE:" + "\t" + r"{:.2f}".format(MAPE(true_v,itervp_CNN_2x256_2[-1])))


src_x,src_z,rcv_x,rcv_z = get_observed_sys(shot_num=10,rcv_num=20)
axes21.scatter(rcv_x,rcv_z+0.02,c='w',marker='v',s=10)
axes21.scatter(src_x,src_z+0.02,c='r',marker='*',s=100)
axes22.scatter(rcv_x,rcv_z+0.02,c='w',marker='v',s=10)
axes22.scatter(src_x,src_z+0.02,c='r',marker='*',s=100)

axes21.set_xlabel("Distance (km)", fontsize=14)
axes22.set_xlabel("Distance (km)", fontsize=14)
axes21.set_ylabel("Depth (km)", fontsize=14)
axes22.set_ylabel("Depth (km)", fontsize=14)

from matplotlib.ticker import MaxNLocator
axes21.xaxis.set_major_locator(MaxNLocator(7))
axes22.xaxis.set_major_locator(MaxNLocator(7))

fig.text(0.1, 0.89, "(a)", fontsize=14, fontweight='bold', ha="center", va="center")
fig.text(0.1, 0.62, "(b)", fontsize=14, fontweight='bold', ha="center", va="center")
fig.text(0.12, 0.31, "(c)", fontsize=14, fontweight='bold', ha="center", va="center")
fig.text(0.485, 0.31, "(d)", fontsize=14, fontweight='bold', ha="center", va="center")

# Adjust layout for better spacing
plt.subplots_adjust(hspace=0.3, wspace=0.1, right=0.85)


import matplotlib as mpl
def add_bottom_cax(ax, pad, height,shrink=1):
    axpos = ax.get_position()
    width = axpos.x1 - axpos.x0
    left_position = axpos.x0 + width * (1 - shrink) / 2
    caxpos = mpl.transforms.Bbox.from_extents(
        left_position,
        axpos.y0 - pad,
        left_position + width * shrink,
        axpos.y0 - pad + height
    )
    cax = ax.figure.add_axes(caxpos)
    return cax

# add horizontal colorbar for [axes21, axes22]
cbar_ax1 = add_bottom_cax(axes21, 0.08, 0.02,shrink=0.95)
cbar1 = fig.colorbar(im1, cax=cbar_ax1, orientation='horizontal', pad=0.1, shrink=0.8)
cbar1.ax.tick_params(labelsize=14)
cbar1.ax.text(1.02, 0.5, r'$m/s$', fontsize=14, transform=cbar1.ax.transAxes, 
              verticalalignment='center', horizontalalignment='left')

cbar_ax2 = add_bottom_cax(axes22, 0.08, 0.02,shrink=0.95)
cbar2 = fig.colorbar(im2, cax=cbar_ax2, orientation='horizontal', pad=0.1, shrink=0.8)
cbar2.ax.tick_params(labelsize=14)
cbar2.ax.text(1.02, 0.5, r'$m/s$', fontsize=14, transform=cbar2.ax.transAxes, 
              verticalalignment='center', horizontalalignment='left')

# plt.savefig("./Figures/Figure7_Imputation_Tests_on_Marmousi2.png",bbox_inches='tight',dpi=300)
# plt.savefig("./Figures_PDF/Figure7_Imputation_Tests_on_Marmousi2.pdf",bbox_inches='tight',dpi=300,format="pdf")
plt.savefig("./Figures_SVG/Figure7_Imputation_Tests_on_Marmousi2.svg",bbox_inches='tight',format="svg")

plt.show()
